# FoldMason Toolkit

![FoldMason](https://proto-bio.github.io/proto-assets/images/tool/foldmason/hero.png)

This notebook demonstrates the two FoldMason tools:

1. **`foldmason-msa`** — multiple structure alignment (remote default; tuneable local mode)
2. **`foldmason-score-msa`** — score an existing MSA with average + per-column LDDT (local-only)

Reference: Gilchrist et al., *Science* 2026 ([DOI](https://doi.org/10.1126/science.ads6733)).

In [1]:
import requests
from proto_tools.tools.structure_alignment import (
    FoldmasonMSAConfig, FoldmasonMSAInput, run_foldmason_msa,
    FoldmasonScoreMSAConfig, FoldmasonScoreMSAInput, run_foldmason_score_msa,
)
from proto_tools.utils.notebook_docs import display_api_reference

## Step 1: fetch a TIM-barrel structural family

We'll align three TIM-barrel triosephosphate isomerases from different organisms — a classic example of structurally similar enzymes spanning broad sequence divergence.

In [2]:
tim_pdbs = {}
for pdb_id in ("1TIM", "8TIM", "1TPF"):
    tim_pdbs[pdb_id] = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=30).text
structures = list(tim_pdbs.values())
ids = list(tim_pdbs.keys())
print(f"Fetched {len(structures)} TIM-barrel structures: {ids}")

Fetched 3 TIM-barrel structures: ['1TIM', '8TIM', '1TPF']


## Step 2: align with `foldmason-msa` (local)

Runs the FoldMason CLI in an isolated standalone env. We use local mode here so the next step (`foldmason-score-msa`, which is local-only) can resolve each MSA row to its structure — the public server's MSA naming differs from the local createdb naming for multi-chain inputs.

In [3]:
# Input fields for run_foldmason_msa
display_api_reference("foldmason", "input", "run_foldmason_msa")

**Input** — `FoldmasonMSAInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structures</code> | <code>list[Structure]</code> | required | Structures to align (Structure objects, file paths, or raw PDB/CIF strings; ≥2) |
| <code>structure_ids</code> | <code>list[str] &#124; None</code> | <code>None</code> | Optional IDs per structure (default: 'structure_0', 'structure_1', ...) |

In [4]:
# Config fields for run_foldmason_msa
display_api_reference("foldmason", "config", "run_foldmason_msa")

**Config** — `FoldmasonMSAConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>search_mode</code> | <code>Literal['remote', 'local']</code> | <code>'remote'</code> | 'remote' (search.foldseek.com/foldmason) or 'local' (foldmason easy-msa) |
| <code>poll_interval_seconds</code> | <code>float</code> | <code>5.0</code> | Remote-only — delay between status polls |
| <code>timeout_seconds</code> | <code>float</code> | <code>600.0</code> | Remote-only — max wall-clock time for the alignment |
| <code>gap_open</code> | <code>int</code> | <code>10</code> | Local-only — gap open cost |
| <code>gap_extend</code> | <code>int</code> | <code>1</code> | Local-only — gap extension cost |
| <code>refine_iters</code> | <code>int</code> | <code>0</code> | Local-only — number of alignment-refinement iterations |
| <code>precluster</code> | <code>bool</code> | <code>False</code> | Local-only — pre-cluster structures before MSA (recommended for large datasets) |
| <code>guide_tree_newick</code> | <code>str &#124; None</code> | <code>None</code> | Local-only — Newick guide tree to use instead of computing one; leaf labels must match structure_ids |
| <code>num_threads</code> | <code>int</code> | <code>4</code> | Local-only — CPU threads |

In [5]:
# Output fields for run_foldmason_msa
display_api_reference("foldmason", "output", "run_foldmason_msa")

**Output** — `FoldmasonMSAOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>ticket_id</code> | <code>str</code> | required | FoldMason job ticket ID (remote only; empty in local mode) |
| <code>aa_msa_fasta</code> | <code>str</code> | required | Amino-acid MSA in FASTA format |
| <code>three_di_msa_fasta</code> | <code>str</code> | required | 3Di-alphabet MSA in FASTA format |
| <code>newick_tree</code> | <code>str</code> | required | Newick guide tree |
| <code>num_sequences</code> | <code>int</code> | required | Number of aligned sequences |
| <code>alignment_length</code> | <code>int</code> | required | Number of MSA columns |
| <code>result_url</code> | <code>str</code> | required | FoldMason result archive URL (remote only) |

In [6]:
msa_out = run_foldmason_msa(
    FoldmasonMSAInput(structures=structures, structure_ids=ids),
    FoldmasonMSAConfig(search_mode="local", num_threads=2),
)
print(f"Aligned {msa_out.num_sequences} structures across {msa_out.alignment_length} columns")
print(f"Guide tree: {msa_out.newick_tree.strip()}")
print()
for line in msa_out.aa_msa_fasta.splitlines()[:6]:
    print(line[:100] + (" ..." if len(line) > 100 else ""))

Running run_foldmason_msa [00:00]

Aligned 6 structures across 251 columns
Guide tree: ((1TPF_A,1TPF_B),(1TIM_A,(1TIM_B,(8TIM_A,8TIM_B))));

>8TIM_A
AP-RKFFVGGNWKMNGDKKSLGELIHTLNGAKLSADTEVVCGAPSIYLDFARQKL-DAKIGVAAQNCYKVPKGAFTGEISPAMIKDIGAAWVILGHSERR ...
>8TIM_B
AP-RKFFVGGNWKMNGDKKSLGELIHTLNGAKLSADTEVVCGAPSIYLDFARQKL-DAKIGVAAQNCYKVPKGAFTGEISPAMIKDIGAAWVILGHSERR ...
>1TIM_B
AP-RKFFVGGNWKMNGKRKSLGELIHTLDGAKLSADTEVVCGAPSIYLDFARQKL-DAKIGVAAQNCYKVPKGAFTGEISPAMIKDIGAAWVILGHSERR ...


## Step 3: score the alignment with `foldmason-score-msa` (local)

Computes per-column and average LDDT scores against the input structures. Useful as a quality metric or downstream design constraint. Local-only — `msa2lddt` is not exposed by the public server.

In [7]:
# Input fields for run_foldmason_score_msa
display_api_reference("foldmason", "input", "run_foldmason_score_msa")

**Input** — `FoldmasonScoreMSAInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structures</code> | <code>list[Structure]</code> | required | Structures whose order matches the rows of `msa` (Structure / path / PDB or CIF text; ≥2) |
| <code>structure_ids</code> | <code>list[str] &#124; None</code> | <code>None</code> | Optional IDs per structure (default: 'structure_0', ...); must match FASTA headers in `msa` |
| <code>msa</code> | <code>MSA</code> | required | Amino-acid MSA (MSA object or raw FASTA string); row order/IDs match `structures` |

In [8]:
# Config fields for run_foldmason_score_msa
display_api_reference("foldmason", "config", "run_foldmason_score_msa")

**Config** — `FoldmasonScoreMSAConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>pair_threshold</code> | <code>float</code> | <code>0.0</code> | Minimum fraction of pair sub-alignments with LDDT info to score a column (0-1) |
| <code>only_scoring_cols</code> | <code>bool</code> | <code>False</code> | Normalise average LDDT by scoring-column count rather than alignment length |
| <code>guide_tree_newick</code> | <code>str &#124; None</code> | <code>None</code> | Newick guide tree to score against; leaf labels must match structure_ids |
| <code>num_threads</code> | <code>int</code> | <code>4</code> | CPU threads |

In [9]:
# Output fields for run_foldmason_score_msa
display_api_reference("foldmason", "output", "run_foldmason_score_msa")

**Output** — `FoldmasonScoreMSAOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>average_lddt</code> | <code>float</code> | required | Average MSA LDDT score (0-1) |
| <code>columns_considered</code> | <code>int</code> | required | Number of MSA columns scored |
| <code>alignment_length</code> | <code>int</code> | required | Total MSA columns |
| <code>column_scores</code> | <code>list[float]</code> | <code>[]</code> | Per-column LDDT scores |

In [10]:
score_out = run_foldmason_score_msa(
    FoldmasonScoreMSAInput(structures=structures, structure_ids=ids, msa=msa_out.aa_msa_fasta),
    FoldmasonScoreMSAConfig(num_threads=2),
)
print(f"Average MSA LDDT: {score_out.average_lddt:.3f}")
print(f"Columns considered: {score_out.columns_considered}/{score_out.alignment_length}")
print(f"First 10 column scores: {[round(s, 2) for s in score_out.column_scores[:10]]}")

Running run_foldmason_score_msa [00:00]

Average MSA LDDT: 0.924
Columns considered: 251/251
First 10 column scores: [0.59, 0.75, 0.5, 0.89, 0.92, 0.91, 0.93, 0.95, 0.95, 0.96]
